In [4]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

In [2]:
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless") # Run silently
    # options.add_argument("--window-size=1920,1080") # Sometimes helps with rendering
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

In [3]:
def scrape_poems_to_dataframe(start_id, end_id):
    driver = setup_driver()
    poem_data = []
    
    print(f"Starting scrape from ID {start_id} to {end_id}...")
    
    for text_id in range(start_id, end_id + 1):
        # We start with cnd_id=1, but we will check the final URL later
        url = f"https://www.greek-language.gr/digitalResources/literature/tools/concordance/browse.html?cnd_id=1&text_id={text_id}"
        
        try:
            driver.get(url)
            time.sleep(0.1) 
            
            # --- NEW STEP 1: IDENTIFY POET ---
            # Strategy: The browser title usually looks like "Κώστας Βάρναλης : ΠΥΘΜΕΝΕΣ"
            page_title_text = driver.title
            
            if ":" in page_title_text:
                # Split by the first colon to get the name
                poet_name = page_title_text.split(":")[0].strip()
            else:
                # Fallback: Sometimes the name is just the first part of the title
                poet_name = "Unknown"

            # Optional: Try to capture the real cnd_id if the site redirected
            current_url = driver.current_url
            try:
                # Extract 'cnd_id=X' from the current URL
                author_id = current_url.split("cnd_id=")[1].split("&")[0]
            except:
                author_id = "1" # Default

            # --- STEP 2: FIND POEM CONTENT ---
            poem_containers = driver.find_elements(By.CSS_SELECTOR, "div.poem")
            
            if not poem_containers:
                continue

            for poem_div in poem_containers:
                # Get Title
                try:
                    title_elem = poem_div.find_element(By.TAG_NAME, "h3")
                    poem_title = title_elem.text.replace("\n", " ").strip()
                except:
                    poem_title = "Unknown Title"

                # Get Lyrics
                lyric_spans = poem_div.find_elements(By.CSS_SELECTOR, "span.l")
                
                if lyric_spans:
                    lines_list = [span.text.strip() for span in lyric_spans if span.text.strip()]
                    full_lyrics = "\n".join(lines_list)
                    
                    print(f"[ID {text_id}] Found: {poet_name} - {poem_title}")
                    
                    poem_data.append({
                        "Poet": poet_name,        # <--- New Column
                        "Author_ID": author_id,   # <--- New Column
                        "Poem_ID": text_id,
                        "Title": poem_title,
                        "Lyrics": full_lyrics,
                    })

        except Exception as e:
            print(f"[ID {text_id}] Error: {e}")
            continue

    driver.quit()
    return pd.DataFrame(poem_data)

In [ ]:
# --- CONFIGURATION ---
START_ID = 3500 
END_ID = 3600
FILENAME = "greek_poems_fixed.csv" 

# --- RUN AND SAVE ---
df = scrape_poems_to_dataframe(START_ID, END_ID)

# Check if we actually have data
if not df.empty:
    file_exists = os.path.isfile(FILENAME)
    # Save only if there is data
    df.to_csv(FILENAME, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')
    print(f"✅ Success: Appended {len(df)} poems to {FILENAME}")
else:
    print(f"⚠️ Skipped: The DataFrame is empty (No poems found in this ID range).")

Starting scrape from ID 3500 to 3600...
[ID 3500] Found: Διονύσιος Σολωμός - [Εις μεγιστάνα]
[ID 3502] Found: Διονύσιος Σολωμός - [Σχέσις πολύπλοκη]
[ID 3503] Found: Διονύσιος Σολωμός - [Το ψίχαλο]
[ID 3504] Found: Διονύσιος Σολωμός - [Εις άρπαγα]
[ID 3505] Found: Διονύσιος Σολωμός - [Εις ψεύτη]
[ID 3506] Found: Διονύσιος Σολωμός - [Ξερή πολυμάθεια]
[ID 3507] Found: Διονύσιος Σολωμός - [Προς τους Επτανήσιους]
[ID 3508] Found: Διονύσιος Σολωμός - [Εις άσχημον]
[ID 3509] Found: Διονύσιος Σολωμός - [Ενώ ένας συλλογιζότουν ανάποδα]
[ID 3512] Found: Διονύσιος Σολωμός - [Πόθος *]
[ID 3513] Found: Διονύσιος Σολωμός - [Η Άνοιξη του Μεταστάσιου]
[ID 3514] Found: Διονύσιος Σολωμός - [Το Καλοκαίρι του Μεταστάσιου]
[ID 3515] Found: Διονύσιος Σολωμός - [Μετάφραση του τεμαχίου του Μεταστάσιου]
[ID 3516] Found: Διονύσιος Σολωμός - [Μετάφραση της Ωδής του Πετράρχη]
[ID 3517] Found: Διονύσιος Σολωμός - [Μίμηση του τραγουδιού της Δεσδεμόνας]
[ID 3518] Found: Διονύσιος Σολωμός - [Αποσπάσματα της μετάφρασ

In [39]:
df = pd.read_csv("poems.csv")

In [59]:
import pandas as pd
import numpy as np

# 1. Load your CSV
# Use index_col=False to ensure the "Unnamed" column is treated as a regular column for now
df = pd.read_csv("poems.csv", index_col=False)

# 2. Define the rows that need fixing (Row 300 onwards)
# Note: We use >= 300. Adjust to 301 if you strictly meant "after 300".
bad_rows_mask = df.index >= 300

print(f"Fixing {bad_rows_mask.sum()} rows...")

# 3. Perform the Shift (Moving values to the Right)
# We move them in reverse order (Right to Left) so we don't overwrite data we need.

# Move 'Title' content -> 'Lyrics' column
df.loc[bad_rows_mask, 'Lyrics'] = df.loc[bad_rows_mask, 'Title']

# Move 'Poem_ID' content -> 'Title' column
df.loc[bad_rows_mask, 'Title'] = df.loc[bad_rows_mask, 'Poem_ID']

# Move 'Author_ID' content -> 'Poem_ID' column (Because '1306' is actually the Poem ID)
df.loc[bad_rows_mask, 'Poem_ID'] = df.loc[bad_rows_mask, 'Author_ID']

# 4. Handle the 'Author_ID' column
# Since we moved the data out of Author_ID, it is now empty or incorrect.
# If you know the Author ID for these rows (e.g., Valaoritis is usually ID 1 or 4), set it here.
# For now, we set it to NaN or a placeholder.
df.loc[bad_rows_mask, 'Author_ID'] = df.loc[bad_rows_mask, "Poet"]
df.loc[bad_rows_mask, 'Poet'] = df.loc[bad_rows_mask, "Unnamed: 0"]

# 5. Drop the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)

# 6. Verify the Fix
print("\n--- Fixed Data Sample (Row 300+) ---")
print(df.loc[300:305, ['Poet', 'Poem_ID', 'Title', 'Lyrics']].head())

# 7. Save the cleaned file
df.to_csv("greek_poems_fixed.csv", index=False, encoding="utf-8-sig")
print("\nSaved to 'greek_poems_fixed.csv'")

Fixing 2410 rows...

--- Fixed Data Sample (Row 300+) ---
                       Poet Poem_ID                        Title  \
300                     300       1                          350   
301  Αριστοτέλης Βαλαωρίτης     350   Εις τον φίλον μου Πάγκαλον   
302  Αριστοτέλης Βαλαωρίτης     352  [Μίαν ημέραν που δεν είχα…]   
303  Αριστοτέλης Βαλαωρίτης     353                     Παράπονο   
304  Αριστοτέλης Βαλαωρίτης     354                     Το δάκρυ   

                                                Lyrics  
300                         Εις τον φίλον μου Πάγκαλον  
301  Εγόγγυζον οι λαίλαπες, απ’ τα φρικτά των στήθη...  
302  Μίαν ημέραν που δεν είχα άλλο τίποτε να κάμω\n...  
303  Πόσες φορές τα κύματα,\nπου επέφταν αφρισμένα\...  
304  Μη με σπαράζεις, άσπλαχνη,\nμε τα παράπονά σου...  

Saved to 'greek_poems_fixed.csv'


C:\Users\nraze\AppData\Local\Temp\ipykernel_13824\3225028473.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Αριστοτέλης Βαλαωρίτης' '1' '1' ... '1' '1' '1']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[bad_rows_mask, 'Author_ID'] = df.loc[bad_rows_mask, "Poet"]


In [74]:
dffixed = pd.read_csv("greek_poems_fixed.csv")

In [75]:
dffixed

,Poet,Author_ID,Poem_ID,Title,Lyrics
0,Μανόλης Αναγνωστάκης,1,4,Χειμώνας 1942,Ξημέρωσεν ο δείχτης πάλι Κυριακή.\nΕφτά μέρες\...
1,Μανόλης Αναγνωστάκης,1,5,Αναμονή,Πόσα χρόνια να γυρίσει…\nΚι όμως η μυρουδιά τη...
2,Μανόλης Αναγνωστάκης,1,6,Απροσδιόριστη χρονολογία,Αυτή η μέρα πέρασε χωρίς καμιάν απόχρωση\nΤόσο...
3,Μανόλης Αναγνωστάκης,1,7,Θά ’ρθει μια μέρα…,Θά ’ρθει μια μέρα που δε θα ’χουμε πια τί να π...
4,Μανόλης Αναγνωστάκης,1,8,13.12.43,Θυμάσαι που σου ’λεγα: όταν σφυρίζουν τα πλοία...
...,...,...,...,...,...
3147,Διονύσιος Σολωμός,1,3531,Κεφάλαιον [6] Το μέλλοντα γενάμενο παρόν. Η κα...,1. Και εκοίταξα τριγύρου και δεν έβλεπα τίποτε...
3148,Διονύσιος Σολωμός,1,3532,Κεφάλαιον [7] Δε σου δίνω μήτε ένα ψίχαλο,1. Αλλά εκαλοκοίταξα εκείνον τον ύπνο και εκατ...
3149,Διονύσιος Σολωμός,1,3533,Κεφάλαιον [8] Το ζωνάρι,1. Αλλά η μάνα της χώρις να κοιτάξει κατά τη θ...
3150,Διονύσιος Σολωμός,1,3534,Κεφάλαιον [9] Η γυναίκα της Ζάκυθος λαβαίνει τ...,"1. Και εχαθήκανε με τες κάσες, και η γυναίκα μ..."


In [62]:
dffixed.tail(50)

,Poet,Author_ID,Poem_ID,Title,Lyrics
2660,Μίλτος Σαχτούρης,1,2950,Ασάη,Όταν ανέβαινες στο βουνό\nεσύ κατέβαινες στην ...
2661,Μίλτος Σαχτούρης,1,2953,Μενέλαος,Μην το φοβάστε το φεγγάρι με το σίδερο\nείπε τ...
2662,Μίλτος Σαχτούρης,1,2954,Η παρουσία,Στις δώδεκα και μισή\nτη νύχτα\nτην ίδια ώρα κ...
2663,Μίλτος Σαχτούρης,1,2955,ΟΙ γενναίοι,"Είναι γενναίοι, όμως κλαίνε\nπιστεύουνε σαν τα..."
2664,Μίλτος Σαχτούρης,1,2956,Μελαγχολία,…στο πλαϊνό μου διαμέρισμα μεταφέραν συνεχώς τ...
2665,Μίλτος Σαχτούρης,1,2957,Όταν,Όταν κλείνω τα μάτια\nξεκινάει από μακριά\nη α...
2666,Μίλτος Σαχτούρης,1,2958,Τα μαλλιά,Σαν τα φυλλώματα των δέντρων\nείναι τα μαλλιά\...
2667,Μίλτος Σαχτούρης,1,2959,Ο Σκλάβος *,Ο Σκλάβος ήθελε να γίνει αεροπόρος\nκάθε γλυπτ...
2668,Μίλτος Σαχτούρης,1,2960,Franz Kafka,Ο Φραντς Κάφκα ζούσε σ’ ένα μεγάλο υγρό δωμάτι...
2669,Μίλτος Σαχτούρης,1,2961,Ο Kafka και τα ψάρια,Στη φωτογραφία του Κάφκα που έχω κολλήσει\nστο...


In [56]:
dffixed = dffixed.iloc[0:2700]

In [57]:
dffixed

,Poet,Author_ID,Poem_ID,Title,Lyrics
0,Μανόλης Αναγνωστάκης,1,4,Χειμώνας 1942,Ξημέρωσεν ο δείχτης πάλι Κυριακή.\nΕφτά μέρες\...
1,Μανόλης Αναγνωστάκης,1,5,Αναμονή,Πόσα χρόνια να γυρίσει…\nΚι όμως η μυρουδιά τη...
2,Μανόλης Αναγνωστάκης,1,6,Απροσδιόριστη χρονολογία,Αυτή η μέρα πέρασε χωρίς καμιάν απόχρωση\nΤόσο...
3,Μανόλης Αναγνωστάκης,1,7,Θά ’ρθει μια μέρα…,Θά ’ρθει μια μέρα που δε θα ’χουμε πια τί να π...
4,Μανόλης Αναγνωστάκης,1,8,13.12.43,Θυμάσαι που σου ’λεγα: όταν σφυρίζουν τα πλοία...
...,...,...,...,...,...
2695,1,Μίλτος Σαχτούρης,2994,Ιούλιος 1990,"—Ε, Μάρκο Πόλο\nμου φώναζε τότε ο «Χριστός»\nά..."
2696,1,Μίλτος Σαχτούρης,2995,Πόνος και τρόμος,Πόνος\nκαι πάλι πόνος\nτρόμος\nκαι πάλι τρόμος...
2697,1,Μίλτος Σαχτούρης,2996,Η μητέρα,Έψαχνα να βρω το σπίτι μου. Οι δρόμοι ήταν\nγε...
2698,1,Μίλτος Σαχτούρης,2997,Σαν πέτρα,Η Άνοιξη είναι για τους ευτυχισμένους\nτότε έλ...
